Trivial baselines on the analysed primary pins.

Analysed = gold main sources, minus `excluded_sources`, pins in `primary_bins` (configs/scoring.yaml).
Weights = the configured primary estimand (`item_weights`), the same as for every model metric.
Verification: always-"no". Distance: best constant whole-metre answer under the weighted median `log_error`
(oracle: chosen on the test pins, the lowest error any constant can reach).

In [ ]:
import numpy as np
import pandas as pd

from thesis.config import REPO_ROOT, load_config
from thesis.eval.scorer import item_weights, log_error, weighted_median

out_dir = REPO_ROOT / "data" / "dataset_v2"
scoring = load_config("scoring")

selected = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")
gold = selected.merge(pd.read_json(out_dir / "gold" / "is_building.jsonl", lines=True), on="source_id")
excluded = {entry["source_id"] for entry in scoring["excluded_sources"]}
main = gold.loc[gold["is_building"] & (gold["assignment"] == "main") & ~gold["source_id"].isin(excluded)]
analysed = pins.merge(main[["source_id", "box_id", "ecoregion"]], on="source_id").query(
    "bin_id in @scoring['primary_bins']"
)
analysed["inside"] = analysed["bin_id"] < 0
analysed["model"] = "baseline"
weight = item_weights(analysed, scoring["estimand"]).to_numpy()
len(main), len(analysed), int(analysed["inside"].sum())

In [ ]:
# always "no": weighted accuracy (primary estimand) and pooled; balanced accuracy is 0.5 by definition
{
    "estimand": scoring["estimand"],
    "always_no_accuracy_weighted": float((weight * ~analysed["inside"]).sum() / weight.sum()),
    "always_no_accuracy_pooled": float((~analysed["inside"]).mean()),
    "always_no_balanced_accuracy": 0.5,
}

In [ ]:
# best constant distance (whole metres, positive; sign is irrelevant for log_error), weighted median like the models
true_m = analysed["exact_signed_distance_m"].to_numpy()
candidates = np.arange(0, 81)
median_error = [weighted_median(log_error(np.full_like(true_m, c), true_m), weight) for c in candidates]
best = int(candidates[np.argmin(median_error)])
analysed["constant_error"] = log_error(np.full_like(true_m, best), true_m)
best, float(np.min(median_error)), analysed.groupby("bin_id")["constant_error"].median()